# Planilha final dos débitos do Nereu — 3 abas

Gera `docs/debitos_nereu_planilha_final.xlsx` com três abas:

1. **Totais** — total de débitos imputados a Nereu vs. total de débitos notificados.
2. **Processos** — um processo por linha (todos os processos onde Nereu tem débito), com data
   de citação (`Cit_Citacoes.Data_envio_AR`) e eventos da CCD identificados como notificação
   de desconto em folha endereçada ao Nereu.
3. **Débitos** — discriminação linha-a-linha de cada débito, com flag `notificado_no_processo`.

**Critério de "notificado"**: nova varredura sobre **todas** as informações CCD dos processos
do Nereu. Cada PDF é classificado por LLM como notificação de desconto em folha endereçada a
ele (`True`/`False`). Os eventos (`SequencialProcessoEvento`) classificados como `True` viram
coluna da aba **Processos**. Não usa `Exe_DescontoFolha` nem a lista hardcoded de 39.


In [1]:
import locale
locale.setlocale(locale.LC_ALL, 'pt_BR.UTF-8')

from ccd.config import load_env
load_env()

from ccd.db import get_connection
conn = get_connection()

from langchain_openai import AzureChatOpenAI
llm_mini = AzureChatOpenAI(model='gpt-5.4-mini')

CPF_NEREU = '13006444434'

import re
_ILLEGAL_XLSX = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f]')

def _strip_ctrl(v):
    return _ILLEGAL_XLSX.sub('', v) if isinstance(v, str) else v


In [2]:
import sys; print(sys.executable)
  # expect: C:\Users\05911205424\Dev\ccd\.venv\Scripts\python.exe

c:\Users\05911205424\Dev\ccd\.venv\Scripts\python.exe


In [3]:
# Imports da seção: tudo o que as células abaixo precisam.
# Variáveis vêm do prelúdio acima: CPF_NEREU, conn, llm_mini, locale,
# _strip_ctrl (sanitizador de chars de controle do xlsx).
from pathlib import Path

import pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from tqdm import tqdm

from ccd.db import run_query_df
from ccd.pdf import extract_text_from_pdf
from ccd.processo import get_info_file_path


In [4]:
from ccd.config import load_env
load_env(override=True)        # force re-read of .env over the stale value
from ccd.db import get_connection
conn = get_connection()        # rebuild the engine with the corrected host/port
import os
print("HOST =", os.getenv("SQL_SERVER_HOST"), "PORT =", os.getenv("SQL_SERVER_PORT"))
  # expect: HOST = 10.24.0.77   PORT = 59678

HOST = 10.24.0.77 PORT = 59678


In [ ]:
# Cache das citações 5d: se o pickle existir, carrega df_citacoes_raw e PULA as duas
# células seguintes (SQL + extração de PDFs + LLM). Apague o pickle para recalcular.
from pathlib import Path

_ckpt_cit = Path('docs/df_citacoes_raw.pkl')
_cit_cached = _ckpt_cit.exists()
if _cit_cached:
    df_citacoes_raw = pd.read_pickle(_ckpt_cit)
    print(f'Citações: carregado de {_ckpt_cit} ({len(df_citacoes_raw)} linhas) — inferência pulada.')
else:
    print('Citações: pickle não encontrado — vai rodar SQL + extração + LLM.')


In [ ]:
if not _cit_cached:
    # CANDIDATOS a citação: informações de setores DAE* / CCD no processo de ORIGEM
    # dos débitos do Nereu, com resumo contendo 'cita'. A LLM (célula abaixo) filtra
    # para citação de 5 dias endereçada especificamente ao Nereu.
    sql_citacoes_nereu = '''
    WITH processos_origem_nereu AS (
        SELECT DISTINCT pro.IdProcesso, pro.numero_processo, pro.ano_processo
        FROM processo.dbo.Processos pro
        INNER JOIN processo.dbo.Exe_Debito ed ON ed.IdProcessoOrigem = pro.IdProcesso
        INNER JOIN processo.dbo.Exe_DebitoPessoa edp ON edp.IDDebito = ed.IdDebito
        INNER JOIN processo.dbo.GenPessoa gp ON gp.IdPessoa = edp.IDPessoa
        WHERE gp.Documento = :cpf
          AND ed.IdDebitoAnterior IS NULL
    )
    SELECT
        CONCAT(inf.numero_processo, '/', inf.ano_processo) AS processo_origem,
        inf.numero_processo,
        inf.ano_processo,
        inf.setor,
        inf.resumo,
        inf.DataPublicacao,
        inf.data_ultima_atualizacao,
        inf.ordem,
        inf.idInformacao,
        CONCAT(RTRIM(inf.setor), '_', inf.numero_processo, '_', inf.ano_processo,
               '_', RIGHT(CONCAT('0000', inf.ordem), 4), '.pdf') AS arquivo
    FROM processo.dbo.vw_ata_informacao inf
    INNER JOIN processos_origem_nereu pn
        ON pn.numero_processo = inf.numero_processo
       AND pn.ano_processo   = inf.ano_processo
    WHERE (LTRIM(RTRIM(inf.setor)) LIKE 'DAE%' OR LTRIM(RTRIM(inf.setor)) = 'CCD')
      AND inf.resumo LIKE '%cita%'
    ORDER BY inf.numero_processo, inf.ano_processo, inf.DataPublicacao, inf.ordem
    '''

    df_citacoes_raw = run_query_df(sql_citacoes_nereu, conn, cpf=CPF_NEREU)
    df_citacoes_raw['DataPublicacao'] = pd.to_datetime(
        df_citacoes_raw['DataPublicacao'], errors='coerce')
    df_citacoes_raw['data_ultima_atualizacao'] = pd.to_datetime(
        df_citacoes_raw['data_ultima_atualizacao'], errors='coerce')
    df_citacoes_raw['data_citacao_efetiva'] = df_citacoes_raw['DataPublicacao'].fillna(
        df_citacoes_raw['data_ultima_atualizacao'])
    df_citacoes_raw['caminho_arquivo'] = df_citacoes_raw.apply(get_info_file_path, axis=1)

    textos_cit = []
    for caminho in tqdm(df_citacoes_raw['caminho_arquivo'], desc='extraindo PDFs de citação'):
        try:
            textos_cit.append(extract_text_from_pdf(caminho) if caminho.exists() else '')
        except Exception as e:
            textos_cit.append(f'__ERRO_EXTRACAO__ {e}')
    df_citacoes_raw['texto'] = textos_cit

    print(f'Candidatos: {len(df_citacoes_raw)} informações em '
          f'{df_citacoes_raw["processo_origem"].nunique()} processos de origem.')
    df_citacoes_raw.head()


In [ ]:
if not _cit_cached:
    # LLM verifica se cada candidato é uma CITAÇÃO DE 5 DIAS endereçada ao Nereu
    class Citacao5DiasNereu(BaseModel):
        eh_citacao_5_dias_nereu: bool = Field(
            description='True somente se o texto for uma citação com prazo de 5 (cinco) dias '
                        'endereçada a NEREU BATISTA LINHARES (CPF 13006444434). False caso '
                        'contrário (outros prazos, outro tipo de ato, ou destinatário diverso).')
        justificativa: str = Field(
            description='Justificativa curta (1-2 frases, pt-BR) com o que sustenta a decisão '
                        '(prazo identificado, nome do destinatário, tipo do ato).')


    prompt_cit = ChatPromptTemplate.from_messages([
        ('system',
         'Você analisa documentos de um Tribunal de Contas. Sua tarefa é decidir se o texto '
         'da informação processual é uma CITAÇÃO COM PRAZO DE 5 (CINCO) DIAS endereçada '
         'a NEREU BATISTA LINHARES (CPF 13006444434). Responda True somente quando ambas '
         'as condições estiverem presentes: (a) o ato é uma citação com prazo de 5 dias '
         '(pode aparecer como "prazo de 5 dias", "cinco dias", "5 (cinco) dias", "art. 19, '
         'parágrafo único", ou contexto equivalente de pagamento de débito em execução); '
         '(b) o destinatário é Nereu Batista Linhares (não outra pessoa). '
         'Responda em pt-BR.'),
        ('human', 'Texto da informação:\n\n{texto}'),
    ])

    chain_cit = prompt_cit | llm_mini.with_structured_output(schema=Citacao5DiasNereu)

    classif_cit = []
    for _, row in tqdm(df_citacoes_raw.iterrows(), total=len(df_citacoes_raw),
                       desc='classificando citações'):
        texto = (row['texto'] or '').strip()
        if not texto or texto.startswith('__ERRO_EXTRACAO__'):
            classif_cit.append({'eh_citacao_5d': False,
                                'justificativa_cit': '(texto vazio ou erro de extração)'})
            continue
        try:
            r = chain_cit.invoke({'texto': texto[:12000]})
            classif_cit.append({
                'eh_citacao_5d': bool(r.eh_citacao_5_dias_nereu),
                'justificativa_cit': r.justificativa,
            })
        except Exception as e:
            classif_cit.append({'eh_citacao_5d': False,
                                'justificativa_cit': f'__ERRO_LLM__ {str(e)[:200]}'})

    df_class_cit = pd.DataFrame(classif_cit)
    df_citacoes_raw['eh_citacao_5d'] = df_class_cit['eh_citacao_5d'].values
    df_citacoes_raw['justificativa_cit'] = df_class_cit['justificativa_cit'].values
    print(f'Citações de 5 dias para Nereu identificadas pela LLM: '
          f'{int(df_citacoes_raw["eh_citacao_5d"].sum())} de {len(df_citacoes_raw)}')
    df_citacoes_raw[df_citacoes_raw['eh_citacao_5d']].head()
    df_citacoes_raw.to_pickle(_ckpt_cit)
    print(f'Citações: inferência salva em {_ckpt_cit}.')


In [10]:
# Por processo_origem, pegar a citação de 5 dias mais RECENTE (a mais nova é a vigente)
df_cit_validas = df_citacoes_raw[df_citacoes_raw['eh_citacao_5d']].copy()

if df_cit_validas.empty:
    df_citacoes = pd.DataFrame(columns=[
        'processo_origem', 'data_citacao', 'setor_citacao',
        'idInformacao_citacao', 'ordem_citacao', 'qtd_citacoes_validas',
    ])
else:
    idx_mais_recente = (
        df_cit_validas
        .sort_values('data_citacao_efetiva', ascending=False)
        .groupby('processo_origem', as_index=False)
        .head(1)
        .index
    )
    df_mais_recente = df_cit_validas.loc[idx_mais_recente, [
        'processo_origem', 'data_citacao_efetiva', 'setor', 'idInformacao', 'ordem',
    ]].rename(columns={
        'data_citacao_efetiva': 'data_citacao',
        'setor': 'setor_citacao',
        'idInformacao': 'idInformacao_citacao',
        'ordem': 'ordem_citacao',
    })
    qtd_por_processo = (
        df_cit_validas.groupby('processo_origem').size()
        .rename('qtd_citacoes_validas').reset_index()
    )
    df_citacoes = df_mais_recente.merge(qtd_por_processo, on='processo_origem', how='left')

print(f'df_citacoes: {len(df_citacoes)} processos de origem com citação de 5 dias '
      f'identificada para Nereu.')
df_citacoes.head()


df_citacoes: 412 processos de origem com citação de 5 dias identificada para Nereu.


,processo_origem,data_citacao,setor_citacao,idInformacao_citacao,ordem_citacao,qtd_citacoes_validas
0,001028/2021,2025-11-27 11:38:16.227,CCD,3137236,68,2
1,003343/2017,2024-12-09 15:43:56.810,DAE_MANDA,2908984,55,1
2,000393/2016,2024-09-23 15:50:10.267,DAE_MANDA,2863711,51,2
3,008899/2016,2024-09-23 15:50:10.220,DAE_MANDA,2863727,59,2
4,100922/2019,2024-09-23 15:11:44.647,DAE_MANDA,2864714,96,2


In [ ]:
# Cache da classificação CCD: se o pickle existir, carrega df_ccd_infos e PULA as
# células de SQL + extração de PDFs + LLM abaixo. Apague o pickle para recalcular.
_ckpt_ccd = Path('docs/df_ccd_infos.pkl')
_ccd_cached = _ckpt_ccd.exists()
if _ccd_cached:
    df_ccd_infos = pd.read_pickle(_ckpt_ccd)
    print(f'CCD: carregado de {_ckpt_ccd} ({len(df_ccd_infos)} linhas) — inferência pulada.')
else:
    print('CCD: pickle não encontrado — vai rodar SQL + extração + LLM.')


In [ ]:
if not _ccd_cached:
    # Todas as informações CCD dos processos onde Nereu tem débito (não só a última)
    sql_todas_ccd_nereu = '''
    WITH processos_nereu AS (
        SELECT DISTINCT pro.IdProcesso, pro.numero_processo, pro.ano_processo
        FROM processo.dbo.Processos pro
        INNER JOIN processo.dbo.Exe_Debito ed ON ed.IdProcessoExecucao = pro.IdProcesso
        INNER JOIN processo.dbo.Exe_DebitoPessoa edp ON edp.IDDebito = ed.IdDebito
        INNER JOIN processo.dbo.GenPessoa gp ON gp.IdPessoa = edp.IDPessoa
        WHERE gp.Documento = :cpf
          AND ed.IdDebitoAnterior IS NULL
    )
    SELECT
        CONCAT(inf.numero_processo, '/', inf.ano_processo) AS processo,
        inf.numero_processo,
        inf.ano_processo,
        inf.setor,
        inf.ordem,
        inf.data_ultima_atualizacao,
        inf.idInformacao,
        ppe.SequencialProcessoEvento AS evento,
        CONCAT(RTRIM(inf.setor), '_', inf.numero_processo, '_', inf.ano_processo,
               '_', RIGHT(CONCAT('0000', inf.ordem), 4), '.pdf') AS arquivo
    FROM processo.dbo.vw_ata_informacao inf
    INNER JOIN processos_nereu pn
        ON pn.numero_processo = inf.numero_processo
       AND pn.ano_processo   = inf.ano_processo
    LEFT JOIN processo.dbo.Pro_ProcessoEvento ppe ON ppe.idInformacao = inf.idInformacao
    WHERE LTRIM(RTRIM(inf.setor)) = 'CCD' and inf.resumo like '%folha%'
    ORDER BY inf.numero_processo, inf.ano_processo, inf.ordem
    '''

    df_ccd_infos = run_query_df(sql_todas_ccd_nereu, conn, cpf=CPF_NEREU)
    df_ccd_infos['caminho_arquivo'] = df_ccd_infos.apply(get_info_file_path, axis=1)
    print(f'Informações CCD nos processos do Nereu: {len(df_ccd_infos)} '
          f'(em {df_ccd_infos["processo"].nunique()} processos)')
    df_ccd_infos.head()


In [ ]:
if not _ccd_cached:
    # Extrai texto de cada PDF da CCD (uma vez por arquivo)
    textos_ccd = []
    for caminho in tqdm(df_ccd_infos['caminho_arquivo'], desc='extraindo PDFs CCD'):
        try:
            textos_ccd.append(extract_text_from_pdf(caminho) if caminho.exists() else '')
        except Exception as e:
            textos_ccd.append(f'__ERRO_EXTRACAO__ {e}')
    df_ccd_infos['texto'] = textos_ccd

    vazios = (df_ccd_infos['texto'].str.len() == 0).sum()
    erros = df_ccd_infos['texto'].str.startswith('__ERRO_EXTRACAO__').sum()
    print(f'Textos extraídos: {len(df_ccd_infos)} | vazios: {vazios} | erros: {erros}')


In [ ]:
if not _ccd_cached:
    # Classificador binário LLM: é notificação de desconto em folha endereçada ao Nereu?
    class NotificacaoNereu(BaseModel):
        eh_notificacao_desconto_folha_nereu: bool = Field(
            description='True se o texto for uma notificação de desconto em folha '
                        'endereçada a NEREU BATISTA LINHARES (CPF 13006444434).')
        justificativa: str = Field(
            description='Justificativa curta (1-2 frases, pt-BR) da classificação.')


    prompt_notif = ChatPromptTemplate.from_messages([
        ('system',
         'Você está classificando uma informação da CCD (Coordenadoria de Controle de Decisões) '
         'do Tribunal de Contas. Responda True somente se o texto for uma NOTIFICAÇÃO DE '
         'DESCONTO EM FOLHA endereçada especificamente a NEREU BATISTA LINHARES '
         '(CPF 13006444434). Responda False para qualquer outro assunto (deliberação, baixa, '
         'cumprimento, etc.) ou se a notificação for para outra pessoa. Responda em pt-BR.'),
        ('human', 'Texto da informação CCD:\n\n{texto}'),
    ])

    chain_notif = prompt_notif | llm_mini.with_structured_output(schema=NotificacaoNereu)


In [ ]:
if not _ccd_cached:
    # Classifica cada informação CCD com a LLM
    classificacoes = []
    for _, row in tqdm(df_ccd_infos.iterrows(), total=len(df_ccd_infos),
                       desc='classificando informações CCD'):
        texto = (row['texto'] or '').strip()
        if not texto or texto.startswith('__ERRO_EXTRACAO__'):
            classificacoes.append({'eh_notificacao': False,
                                   'justificativa': '(texto vazio ou erro de extração)'})
            continue
        try:
            r = chain_notif.invoke({'texto': texto[:12000]})
            classificacoes.append({
                'eh_notificacao': bool(r.eh_notificacao_desconto_folha_nereu),
                'justificativa': r.justificativa,
            })
        except Exception as e:
            classificacoes.append({'eh_notificacao': False,
                                   'justificativa': f'__ERRO_LLM__ {str(e)[:200]}'})

    df_class = pd.DataFrame(classificacoes)
    df_ccd_infos['eh_notificacao'] = df_class['eh_notificacao'].values
    df_ccd_infos['justificativa']  = df_class['justificativa'].values
    print(f'Informações classificadas como notificação para Nereu: '
          f'{df_ccd_infos["eh_notificacao"].sum()} de {len(df_ccd_infos)}')
    df_ccd_infos[df_ccd_infos['eh_notificacao']].head()
    df_ccd_infos.to_pickle(_ckpt_ccd)
    print(f'CCD: inferência salva em {_ckpt_ccd}.')


In [15]:
# Agrega: para cada processo, lista de eventos da CCD identificados como notificação
df_notif = (
    df_ccd_infos[df_ccd_infos['eh_notificacao']]
    .groupby('processo')
    .agg(
        eventos_notificacao_ccd=('evento', lambda s: sorted(int(x) for x in s.dropna().unique())),
        qtd_notificacoes=('evento', 'count'),
    )
    .reset_index()
)
print(f'Processos com pelo menos uma notificação para Nereu: {len(df_notif)}')
df_notif.head()


Processos com pelo menos uma notificação para Nereu: 41


,processo,eventos_notificacao_ccd,qtd_notificacoes
0,000088/2023,[88],1
1,000101/2022,[76],1
2,000102/2022,[64],1
3,000109/2022,[70],1
4,000111/2022,[74],1


In [26]:
from ccd.db import run_query_df

SQL_DEBITOS_NEREU = """
SELECT
    e.IdDebito AS id_debito,
    (SELECT CONCAT(p.numero_processo, '/', p.ano_processo)
        FROM processo.dbo.Processos p WHERE p.IdProcesso = e.IdProcessoOrigem) AS processo_origem,
    (SELECT CONCAT(p.numero_processo, '/', p.ano_processo)
        FROM processo.dbo.Processos p WHERE p.IdProcesso = e.IdProcessoExecucao) AS processo_execucao,
    etd.Descricao AS tipo_debito,
    esd.DescricaoStatusDivida AS situacao_divida,
    e.valorOriginalDebito AS valor_original,
    processo.dbo.fn_Exe_RetornaValorAtualizado(e.IdDebito) AS valor_atualizado,
    e.dataTransito AS data_transito
FROM processo.dbo.Exe_Debito e
INNER JOIN processo.dbo.Exe_DebitoPessoa edp ON edp.IDDebito = e.IdDebito
INNER JOIN processo.dbo.GenPessoa gp ON gp.IdPessoa = edp.IDPessoa
LEFT JOIN processo.dbo.Exe_TipoDebito etd ON etd.CodigoTipoDebito = e.CodigoTipoDebito
LEFT JOIN processo.dbo.Exe_StatusDivida esd ON esd.CodigoStatusDivida = e.CodigoStatusDivida
WHERE gp.Documento = :cpf
  AND e.IdDebitoAnterior IS NULL
  AND esd.StatusCancelamento IS NULL
"""


def carregar_debitos(cpf: str = CPF_NEREU) -> pd.DataFrame:
    df = run_query_df(SQL_DEBITOS_NEREU, cpf=cpf)
    df['tipo_debito'] = df['tipo_debito'].fillna('(sem tipo)')
    df['situacao_divida'] = df['situacao_divida'].fillna('(sem situação)')
    return df


debitos_nereu = carregar_debitos()
print(f'Débitos do Nereu (sem cancelados): {len(debitos_nereu)} linhas')
debitos_nereu.head()

Débitos do Nereu (sem cancelados): 593 linhas


,id_debito,processo_origem,processo_execucao,tipo_debito,situacao_divida,valor_original,valor_atualizado,data_transito
0,10979,002113/2009,None,Multa,Em Aberto,55.00,127.1034,2014-06-02
1,20010,100007/2019,None,Multa,Em Aberto,9100.00,13158.8295,2020-08-11
2,20750,100897/2019,000462/2026,Multa Cominatória,Em Aberto,0.00,NaN,2020-08-04
3,21483,015971/2017,None,Multa,Em Aberto,8864.15,15426.3771,2020-11-11
4,21475,101908/2018,000137/2022,Multa,Em Aberto,8864.15,15568.8670,2020-10-09


In [33]:
# NOVA — inserir após a célula 11. Metadados por processo: setor atual, número do acórdão
# (mais recente, do processo de origem) e o evento (SequencialProcessoEvento) da citação 5d.
# Define também _fmt_data (usada nas abas para formatar datas como DD/MM/YYYY).
# Consultas baratas (sem LLM). Constrói df_setor, df_acordao e df_citacoes_ev.
from ccd.db import run_query_df


def _fmt_data(s):
    """Formata uma coluna de datas como DD/MM/YYYY (vazio quando NaT)."""
    if not pd.api.types.is_datetime64_any_dtype(s):
        s = pd.to_datetime(s, errors='coerce', format='mixed')
    return s.dt.strftime('%d/%m/%Y').fillna('')


_SUB_EXEC_ORIG = """
    SELECT e.IdProcessoExecucao AS idp FROM processo.dbo.Exe_Debito e
    JOIN processo.dbo.Exe_DebitoPessoa edp ON edp.IDDebito = e.IdDebito
    JOIN processo.dbo.GenPessoa gp ON gp.IdPessoa = edp.IDPessoa
    WHERE gp.Documento = :cpf AND e.IdDebitoAnterior IS NULL
    UNION
    SELECT e.IdProcessoOrigem FROM processo.dbo.Exe_Debito e
    JOIN processo.dbo.Exe_DebitoPessoa edp ON edp.IDDebito = e.IdDebito
    JOIN processo.dbo.GenPessoa gp ON gp.IdPessoa = edp.IDPessoa
    WHERE gp.Documento = :cpf AND e.IdDebitoAnterior IS NULL
"""
_SUB_ORIG = """
    SELECT e.IdProcessoOrigem AS idp FROM processo.dbo.Exe_Debito e
    JOIN processo.dbo.Exe_DebitoPessoa edp ON edp.IDDebito = e.IdDebito
    JOIN processo.dbo.GenPessoa gp ON gp.IdPessoa = edp.IDPessoa
    WHERE gp.Documento = :cpf AND e.IdDebitoAnterior IS NULL
"""

# setor atual (processos de execução + origem)
df_setor = run_query_df(f"""
SELECT DISTINCT CONCAT(p.numero_processo, '/', p.ano_processo) AS processo,
       RTRIM(p.setor_atual) AS setor_atual
FROM processo.dbo.Processos p
WHERE p.IdProcesso IN ({_SUB_EXEC_ORIG})
""", cpf=CPF_NEREU)

# número do acórdão (sessão mais recente) no processo de origem
_ac = run_query_df(f"""
SELECT CONCAT(v.NumeroProcesso, '/', v.AnoProcesso) AS processo_origem,
       v.numeroResultado, v.anoResultado, v.DataSessao
FROM processo.dbo.vw_ia_votos_acordaos_decisoes v
WHERE v.numeroResultado IS NOT NULL
  AND v.IdProcesso IN ({_SUB_ORIG})
""", cpf=CPF_NEREU)
_ac['DataSessao'] = pd.to_datetime(_ac['DataSessao'], errors='coerce')
df_acordao = (
    _ac.sort_values('DataSessao', ascending=False)
       .groupby('processo_origem', as_index=False)
       .head(1)
       .copy()
)
df_acordao['numero_acordao'] = (
    df_acordao['numeroResultado'].astype(str).str.strip() + '/'
    + df_acordao['anoResultado'].astype(str).str.strip()
)
df_acordao = df_acordao.rename(columns={'DataSessao': 'data_acordao'})[
    ['processo_origem', 'numero_acordao', 'data_acordao']]

# evento (SequencialProcessoEvento) da citação 5d, via Pro_ProcessoEvento
_ev = run_query_df(f"""
SELECT ppe.IdInformacao, ppe.SequencialProcessoEvento
FROM processo.dbo.Pro_ProcessoEvento ppe
WHERE ppe.IdInformacao IS NOT NULL
  AND ppe.IdProcesso IN ({_SUB_ORIG})
""", cpf=CPF_NEREU)
_ev = _ev.dropna(subset=['IdInformacao']).drop_duplicates('IdInformacao')
_ev['IdInformacao'] = _ev['IdInformacao'].astype('int64')

df_citacoes_ev = df_citacoes.copy()
df_citacoes_ev['idInformacao_citacao'] = df_citacoes_ev['idInformacao_citacao'].astype('int64')
df_citacoes_ev = df_citacoes_ev.merge(
    _ev.rename(columns={'IdInformacao': 'idInformacao_citacao',
                        'SequencialProcessoEvento': 'evento_citacao_5d'}),
    on='idInformacao_citacao', how='left')

print(f'setor: {len(df_setor)} | acórdãos: {len(df_acordao)} | '
      f'citações c/ evento: {df_citacoes_ev["evento_citacao_5d"].notna().sum()}/{len(df_citacoes_ev)}')
df_citacoes_ev.head()


setor: 1058 | acórdãos: 591 | citações c/ evento: 411/412


,processo_origem,data_citacao,setor_citacao,idInformacao_citacao,ordem_citacao,qtd_citacoes_validas,evento_citacao_5d
0,001028/2021,2025-11-27 11:38:16.227,CCD,3137236,68,2,110.0
1,003343/2017,2024-12-09 15:43:56.810,DAE_MANDA,2908984,55,1,96.0
2,000393/2016,2024-09-23 15:50:10.267,DAE_MANDA,2863711,51,2,90.0
3,008899/2016,2024-09-23 15:50:10.220,DAE_MANDA,2863727,59,2,100.0
4,100922/2019,2024-09-23 15:11:44.647,DAE_MANDA,2864714,96,2,131.0


In [41]:
# SUBSTITUI a célula 12 (Aba 1 — Totais). Inalterada; opera sobre debitos_nereu sem cancelados.
processos_notificados = set(df_notif['processo'])
mask_notif = debitos_nereu['processo_execucao'].isin(processos_notificados)

qtd_total = len(debitos_nereu)
qtd_notif = int(mask_notif.sum())
val_total = float(debitos_nereu['valor_atualizado'].sum())
val_notif = float(debitos_nereu.loc[mask_notif, 'valor_atualizado'].sum())

tab_totais = pd.DataFrame([
    {'metrica': 'Total de débitos imputados a Nereu',
     'quantidade': qtd_total,
     'valor_atualizado_R$': locale.currency(val_total, grouping=True, symbol=True)},
    {'metrica': 'Total de débitos notificados (desconto em folha, Nereu)',
     'quantidade': qtd_notif,
     'valor_atualizado_R$': locale.currency(val_notif, grouping=True, symbol=True)},
])
tab_totais


,metrica,quantidade,valor_atualizado_R$
0,Total de débitos imputados a Nereu,593,"R$ 4.870.929,19"
1,Total de débitos notificados (desconto em folh...,41,"R$ 415.405,29"


In [42]:
# SUBSTITUI a célula 13 (Aba 2 — Processos). Grão: 1 linha por par (execução, origem).
base = (
    debitos_nereu.groupby(['processo_execucao', 'processo_origem'], dropna=False)
    .agg(
        valor_original=('valor_original', 'sum'),
        valor_atualizado=('valor_atualizado', 'sum'),
        data_transito=('data_transito', 'max'),
    )
    .reset_index()
)

# setor atual: do processo de EXECUÇÃO; se só houver origem (sem execução), usa o da ORIGEM
_setor_map = df_setor.drop_duplicates('processo').set_index('processo')['setor_atual']
base['setor_atual'] = (base['processo_execucao'].map(_setor_map)
                       .fillna(base['processo_origem'].map(_setor_map)))
# acórdão mais recente (processo de origem)
base = base.merge(df_acordao, on='processo_origem', how='left')
# citação 5d + evento (processo de origem)
base = base.merge(
    df_citacoes_ev[['processo_origem', 'data_citacao', 'setor_citacao',
                    'evento_citacao_5d', 'qtd_citacoes_validas']],
    on='processo_origem', how='left')
# notificações CCD (processo de execução)
base = base.merge(
    df_notif[['processo', 'qtd_notificacoes', 'eventos_notificacao_ccd']]
        .rename(columns={'processo': 'processo_execucao'}),
    on='processo_execucao', how='left')

base['qtd_notificacoes'] = base['qtd_notificacoes'].fillna(0).astype(int)
base['qtd_citacoes_validas'] = base['qtd_citacoes_validas'].fillna(0).astype(int)
base['evento_citacao_5d'] = base['evento_citacao_5d'].astype('Int64')
base['eventos_notificacao_ccd'] = base['eventos_notificacao_ccd'].apply(
    lambda v: v if isinstance(v, list) else [])

tab_processos = base[[
    'processo_execucao', 'processo_origem', 'setor_atual',
    'numero_acordao', 'data_acordao', 'data_transito',
    'data_citacao', 'evento_citacao_5d', 'setor_citacao',
    'valor_original', 'valor_atualizado',
    'qtd_citacoes_validas', 'qtd_notificacoes', 'eventos_notificacao_ccd',
]].sort_values('valor_atualizado', ascending=False).copy()

for _c in ('data_acordao', 'data_transito', 'data_citacao'):
    tab_processos[_c] = _fmt_data(tab_processos[_c])

print(f'tab_processos: {len(tab_processos)} pares (execução, origem)')
tab_processos.head()


tab_processos: 590 pares (execução, origem)


,processo_execucao,processo_origem,setor_atual,numero_acordao,data_acordao,data_transito,data_citacao,evento_citacao_5d,setor_citacao,valor_original,valor_atualizado,qtd_citacoes_validas,qtd_notificacoes,eventos_notificacao_ccd
465,NaN,001028/2021,CCD,111/2022,15/06/2022,18/11/2022,27/11/2025,110,CCD,45955.47,58782.6684,2,0,[]
178,001361/2022,004784/2013,PROC_EXE,1262/2015,28/07/2015,26/12/2016,,<NA>,NaN,17728.31,38772.1474,0,0,[]
418,003654/2022,011038/2014,CCD,2966/2016,12/07/2016,13/12/2016,,<NA>,NaN,17728.31,35541.0532,0,0,[]
538,NaN,016175/2015,ARQUI,823/2019,07/11/2019,17/12/2019,19/05/2022,67,DAE_MANDA,17728.31,32263.3106,1,0,[]
507,NaN,009854/2017,ARQUI,805/2019,05/11/2019,13/12/2019,31/05/2022,78,DAE_MANDA,17728.31,32231.3488,1,0,[]


In [43]:
# SUBSTITUI a célula 14 (Aba 3 — Débitos). Sem a coluna notificado_no_processo.
tab_debitos = debitos_nereu[[
    'id_debito', 'processo_origem', 'processo_execucao',
    'tipo_debito', 'situacao_divida',
    'valor_original', 'valor_atualizado', 'data_transito',
]].copy().sort_values('valor_atualizado', ascending=False)
tab_debitos['data_transito'] = _fmt_data(tab_debitos['data_transito'])
tab_debitos.head()


,id_debito,processo_origem,processo_execucao,tipo_debito,situacao_divida,valor_original,valor_atualizado,data_transito
63,22068,004784/2013,001361/2022,Multa,Em Aberto,17728.31,38772.1474,26/12/2016
444,28580,001028/2021,None,Multa,Em Aberto,30000.00,35959.3581,18/11/2022
143,22514,011038/2014,003654/2022,Multa,Em Aberto,17728.31,35541.0532,13/12/2016
93,22516,016175/2015,None,Multa,Em Aberto,17728.31,32263.3106,17/12/2019
98,22524,009854/2017,None,Multa,Em Aberto,17728.31,32231.3488,13/12/2019


In [44]:
# NOVA — Aba 4 (Notificações desconto em folha): 1 linha por evento de notificação.
# Inclui a data em que a CCD fez a informação de notificação (data_ultima_atualizacao).
# Débitos do processo (execução) agregados em lista (não há vínculo direto evento->débito).
_notif_eventos = (
    df_ccd_infos[df_ccd_infos['eh_notificacao']]
    [['processo', 'evento', 'data_ultima_atualizacao']]
    .drop_duplicates()
    .rename(columns={'processo': 'processo_execucao',
                     'data_ultima_atualizacao': 'data_notificacao'})
)

_deb_por_proc = (
    debitos_nereu.groupby('processo_execucao')
    .agg(
        ids_debitos=('id_debito', lambda s: sorted(int(x) for x in s)),
        valor_original_total=('valor_original', 'sum'),
        valor_atualizado_total=('valor_atualizado', 'sum'),
    )
    .reset_index()
)

tab_notificacoes = (
    _notif_eventos.merge(_deb_por_proc, on='processo_execucao', how='left')
    .rename(columns={'processo_execucao': 'processo'})
)
tab_notificacoes['ids_debitos'] = tab_notificacoes['ids_debitos'].apply(
    lambda v: v if isinstance(v, list) else [])
tab_notificacoes['valor_original_total'] = tab_notificacoes['valor_original_total'].fillna(0.0)
tab_notificacoes['valor_atualizado_total'] = tab_notificacoes['valor_atualizado_total'].fillna(0.0)
tab_notificacoes = tab_notificacoes[[
    'processo', 'evento', 'data_notificacao', 'ids_debitos',
    'valor_original_total', 'valor_atualizado_total',
]].sort_values(['processo', 'evento']).reset_index(drop=True)
tab_notificacoes['data_notificacao'] = _fmt_data(tab_notificacoes['data_notificacao'])

# Dados do PROCESSO (débitos e valores) só na 1ª linha (menor evento) de cada processo
# -> somar valor_atualizado_total passa a bater com o total notificado da aba Totais.
_primeira = ~tab_notificacoes.duplicated('processo')
for _c in ('ids_debitos', 'valor_original_total', 'valor_atualizado_total'):
    tab_notificacoes[_c] = tab_notificacoes[_c].where(_primeira)
print(f"soma valor_atualizado_total (1x/processo): "
      f"{tab_notificacoes['valor_atualizado_total'].sum():.2f}")
print(f'Notificações (eventos): {len(tab_notificacoes)} | '
      f'processos: {tab_notificacoes["processo"].nunique()}')
tab_notificacoes.head()


soma valor_atualizado_total (1x/processo): 415405.29
Notificações (eventos): 42 | processos: 41


,processo,evento,data_notificacao,ids_debitos,valor_original_total,valor_atualizado_total
0,000088/2023,88,15/09/2025,[22834],1000.00,1439.7034
1,000101/2022,76,15/09/2025,[27505],8864.15,8985.1035
2,000102/2022,64,27/03/2026,[27506],8864.15,9533.0507
3,000109/2022,70,27/03/2026,[27517],8027.40,8425.8962
4,000111/2022,74,15/09/2025,[21681],8864.15,15214.2555


In [45]:
# SUBSTITUI a célula 15 (escrita do xlsx) — agora com 4 abas. Datas já em DD/MM/YYYY.
out_xlsx = Path('docs/debitos_nereu_planilha_final.xlsx')
out_xlsx.parent.mkdir(parents=True, exist_ok=True)


def _sanitize(df):
    return df.map(_strip_ctrl) if hasattr(df, 'map') else df.applymap(_strip_ctrl)


# colunas-lista -> string (openpyxl não aceita listas)
tab_processos_xlsx = tab_processos.copy()
tab_processos_xlsx['eventos_notificacao_ccd'] = tab_processos_xlsx['eventos_notificacao_ccd'].apply(
    lambda v: ', '.join(str(x) for x in v) if isinstance(v, list) and v else '')

tab_notificacoes_xlsx = tab_notificacoes.copy()
tab_notificacoes_xlsx['ids_debitos'] = tab_notificacoes_xlsx['ids_debitos'].apply(
    lambda v: ', '.join(str(x) for x in v) if isinstance(v, list) and v else '')

with pd.ExcelWriter(out_xlsx, engine='openpyxl') as writer:
    _sanitize(tab_totais).to_excel(writer, sheet_name='Totais', index=False)
    _sanitize(tab_processos_xlsx).to_excel(writer, sheet_name='Processos', index=False)
    _sanitize(tab_debitos).to_excel(writer, sheet_name='Débitos', index=False)
    _sanitize(tab_notificacoes_xlsx).to_excel(
        writer, sheet_name='Notificações desconto em folha', index=False)

print(f'Planilha salva em: {out_xlsx.resolve()}')


Planilha salva em: C:\Users\05911205424\Dev\ccd\scripts\analise\docs\debitos_nereu_planilha_final.xlsx
